In [30]:
# --- 1. Librerías ---
import pandas as pd
import statsmodels.formula.api as smf

# --- 2. Cargar datos ---
url = "https://raw.githubusercontent.com/LOST-STATS/LOST-STATS.github.io/master/Model_Estimation/Data/Event_Study_DiD/bacon_example.csv"
df = pd.read_csv(url)

In [40]:
# ===============================
# 0. Librerías
# ===============================
import pandas as pd
import statsmodels.formula.api as smf

# ===============================
# 1. Limpiar nombres de columnas
# ===============================
df.columns = df.columns.str.strip()

# ===============================
# 2. Definir identificadores y variables de tratamiento
# ===============================
df['id'] = df['stfips']        # Identificador de unidad
df['period'] = df['year']      # Periodo temporal
df['treated'] = (df['_nfd'] == 1).astype(int)  # 1 = tratado, 0 = no tratado

# ===============================
# 3. Primer período de tratamiento por unidad
# ===============================
def first_treatment(g):
    treated_periods = g.loc[g['treated']==1, 'period']
    if len(treated_periods) > 0:
        return treated_periods.min()
    else:
        return pd.NA  # nunca tratado

# Asignar G a cada fila según id
df['G'] = df.groupby('id').apply(first_treatment).reindex(df['id']).values

# ===============================
# 4. Tiempo relativo al tratamiento
# ===============================
df['event_time'] = df['period'] - df['G']

# ===============================
# 5. Crear dummies de tiempo relativo (excluimos referencia k=-1)
# ===============================
dummies = pd.get_dummies(df['event_time'], prefix='k')
if 'k_-1' in dummies.columns:
    dummies.drop('k_-1', axis=1, inplace=True)
df = pd.concat([df, dummies], axis=1)

# Lista de columnas de dummies para TWFE
event_time_cols = [col for col in df.columns if col.startswith('k_')]

# ===============================
# 6. Fórmula TWFE para estimar efectos dinámicos
# outcome: asmrs
# controles: pcinc, asmrh, cases
# efectos fijos: unidad y periodo
# ===============================
formula = 'asmrs ~ ' + ' + '.join(event_time_cols) + ' + pcinc + asmrh + cases + C(id) + C(period)'

# ===============================
# 7. Estimación OLS con errores clusterizados por unidad
# ===============================
model = smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['id']})

C:\Users\Matias\AppData\Local\Temp\ipykernel_6372\3991641571.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['G'] = df.groupby('id').apply(first_treatment).reindex(df['id']).values


In [43]:
# ===============================
# 8. Extraer coeficientes de dummies (ATT dinámicos)
# ===============================
# Filtrar nombres de coeficientes que empiezan con 'k_'
att_dynamic_cols = [c for c in model.params.index if c.startswith('k_')]

att_dynamic = model.params[att_dynamic_cols]
se_dynamic = model.bse[att_dynamic_cols]

# ===============================
# 9. Crear tabla limpia de ATT(g,t)
# ===============================
def extract_event_time(name):
    # nombre tipo 'k_1964[T.True]' -> extraer '1964'
    return int(name.split('_')[1].split('[')[0])

att_table = pd.DataFrame({
    'event_time': [extract_event_time(c) for c in att_dynamic_cols],
    'ATT': att_dynamic.values,
    'SE': se_dynamic.values
}).sort_values('event_time')

print(att_table)



    event_time       ATT        SE
0         1964 -7.881628  4.651220
1         1965 -0.639113  2.065135
2         1966 -1.893366  2.084196
3         1967 -0.689633  1.575349
4         1968 -0.476271  1.652736
5         1969  1.389956  1.223285
6         1970  2.445805  1.141058
7         1971  5.997670  1.301433
8         1972  4.148986  1.045389
9         1973  5.170411  0.883660
10        1974  5.385563  0.861984
11        1975  7.004214  0.978774
12        1976  5.195175  0.958779
13        1977  5.902253  0.799318
14        1978  4.121173  0.736560
15        1979  2.976472  0.952145
16        1980  1.427207  0.640890
17        1981  2.281924  0.862794
18        1982  0.886151  0.714611
19        1983  0.355690  0.877914
20        1984  1.684953  0.848177
21        1985 -0.168447  1.316127
22        1986  2.202552  1.156733
23        1987  2.792193  1.150986
24        1988  1.473182  1.488848
25        1989  1.110153  1.724074
26        1990  1.373999  1.651545
27        1991  0.85